# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

## Createing state lookup tables

For every citation we need to know both the state of the citing patent and the state of the cited patent. 
I have created two small DataFrames from the patent dataset, one keyed by CITING and one keyed by CITED.

In [22]:
cited_states = patents.select(
    col("PATENT").alias("CITED"),
    col("POSTATE").alias("CITED_STATE"))

citing_states = patents.select(
    col("PATENT").alias("CITING"),
    col("POSTATE").alias("CITING_STATE"))

## Added the cited and citing patent states

I have joined the citation records with the patent dataset twice. The first join finds the state of the cited patent and the second join finds the state of the citing patent. 

I have used left join because not every patent appearing in the citation dataset necessarily appears in the patent dataset.

In [23]:
citation_states = (
    citations
    .join(cited_states, on="CITED", how="left")
    .join(citing_states, on="CITING", how="left")
    .select("CITING","CITED","CITING_STATE","CITED_STATE"))

In [10]:
citation_states.show(20)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|1331793|       NULL|3858258|          CA|
|1540798|       NULL|3858258|          CA|
| 924225|       NULL|3858527|        NULL|
|3638586|         CA|3858527|        NULL|
|2444326|       NULL|3858527|        NULL|
|3699902|         OH|3858527|        NULL|
|2705120|       NULL|3858527|        NULL|
|2967080|       NULL|3858527|        NULL|
|3602157|         TX|3858527|        NULL|
| 957631|       NULL|3858560|          IN|
|3815160|         NY|3858597|          MT|
|3675252|         AZ|3858597|          MT|
|2777621|       NULL|3858770|          CA|
|2290722|       NULL|3858770|          CA|
|2782969|       NULL|3858770|          CA|
|3040941|       NULL|3858770|          CA|
| 982044|       NULL|3859029|          NY|
|1830227|       NULL|3859029|          NY|
|2752631|       NULL|3859029|          NY|
|3741706|         OH|3859029|          NY|
+-------+--

## Identify same-state citations


In [11]:
same_state_citations = citation_states.filter(
    col("CITED_STATE").isNotNull() &
    col("CITING_STATE").isNotNull() &
    (col("CITED_STATE") == col("CITING_STATE")))

In [12]:
same_state_citations.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|3368197|         MI|3859627|          MI|
|3722929|         CA|3860191|          CA|
|3172282|         AZ|3861180|          AZ|
|3791450|         MA|3861473|          MA|
|3802510|         MA|3861473|          MA|
|3118651|         MI|3862577|          MI|
|3099569|         PA|3862844|          PA|
|3769543|         NY|3863090|          NY|
|3467396|         MI|3863935|          MI|
|3167490|         NY|3864160|          NY|
+-------+-----------+-------+------------+
only showing top 10 rows



## Counting same-state citations for each citing patent

The remaining rows represent valid same-state citations. I group them by the citing patent and count the number of matching citation records.

In [13]:
same_state_counts = (same_state_citations.groupBy("CITING").agg(count("*").alias("SAME_STATE")))

In [15]:
same_state_counts.show(15)

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|3859627|         1|
|3860191|         1|
|3861180|         1|
|3861473|         2|
|3862577|         1|
|3862844|         1|
|3863090|         1|
|3863935|         1|
|3864160|         1|
|3864244|         2|
|3864539|         1|
|3864676|         3|
|3865697|         1|
|3866515|         2|
|3869420|         1|
+-------+----------+
only showing top 15 rows



## changes in the original patent dataset

I left join the calculated same-state counts back to the complete patent
dataset. Missing counts are replaced with zero.

In [16]:
result = (patents.join(same_state_counts,patents.PATENT == same_state_counts.CITING,"left").drop("CITING")
          .fillna({"SAME_STATE": 0}))

In [17]:
result.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    63| NULL|       9|    NULL| 0.3704|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|         0|
|3070805| 1963| 1096|   NULL|     US|     CA|    NULL|      1|  NULL|     2|  6|    63| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|         0|
|3070811| 1963| 1096|   NULL| 

In [18]:
result.filter(col("PATENT") == 6009554).select("PATENT","POSTATE","CMADE","SAME_STATE").show()

+-------+-------+-----+----------+
| PATENT|POSTATE|CMADE|SAME_STATE|
+-------+-------+-----+----------+
|6009554|     NY|    9|         8|
+-------+-------+-----+----------+



In [24]:
citation_states.filter(
    col("CITING") == 6009554
).show(20, truncate=False)

+-------+-------+------------+-----------+
|CITING |CITED  |CITING_STATE|CITED_STATE|
+-------+-------+------------+-----------+
|6009554|4029274|NY          |NY         |
|6009554|4831521|NY          |NY         |
|6009554|4181849|NY          |NY         |
|6009554|4494717|NY          |NULL       |
|6009554|4611291|NY          |NY         |
|6009554|4617662|NY          |NY         |
|6009554|4740972|NY          |NY         |
|6009554|5048064|NY          |NY         |
|6009554|5364047|NY          |NY         |
+-------+-------+------------+-----------+



## ten patents with the most same-state citations


In [19]:
top_dataframe = result.orderBy(col("SAME_STATE").desc())

top_dataframe.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 